# Coherent Flare NX am LAPIS-Aufbau testen

Im Modus 3 kann jede passende Flanke am Triggereingang einen Laserpuls auslösen. Der externe Sicherheitsinterlock muss vor dem Scharfstellen am Aufbau freigegeben sein. Für den ersten Statustest den Interlock offen lassen. Für einen reinen Offline-Test nur die Konfigurationszelle und die Zelle „Treiber ohne Hardware prüfen“ ausführen.

Die Konfigurationszelle lädt den Lasertreiber und seine Tests neu. Nach Änderungen am Lasertreiber diese Zelle erneut ausführen. Vor einem Hardwaretest die Gerätezuordnung prüfen und andere Programme am selben COM-Port schließen. Jeder Laser-Test öffnet eine eigene Verbindung. Beim Verbinden wird der Handshake gesetzt. Beim Schließen übernimmt der Treiber bei Bedarf die serielle Steuerung von REM, sendet OFF und prüft die Rückmeldung.

In [1]:
from importlib import reload
from pathlib import Path
import sys
import time
import unittest

roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "LAPIS-Control-Suite"]
project = next((p for p in roots if (p / "pyproject.toml").is_file()
                and (p / "src/lapis_control").is_dir()), None)
if project is None:
    raise FileNotFoundError("Den Kernel im LAPIS-Projekt starten.")
src = str(project / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from lapis_control.configuration import load_devices
from lapis_control.devices.laser import flare_nx
from lapis_control.devices.laser.tests import test_flare_nx

if "laser" in globals() and laser.is_connected:
    raise RuntimeError("Die laufende Laserverbindung vor der Neukonfiguration schließen.")
reload(flare_nx)
reload(test_flare_nx)

CONFIG_PATH = project / "config/devices.example.json"  # Ggf. devices.local.json wählen.
devices = load_devices(CONFIG_PATH)
laser = devices["laser"]
print(f"Gewählte Konfiguration: {CONFIG_PATH.name} · Laser: {laser.port} · Handshake: {laser.handshake_interface}")

Gewählte Konfiguration: devices.example.json · Laser: COM10 · Handshake: USB


## Treiber ohne Hardware prüfen

Die Tests simulieren die seriellen Antworten, einschließlich Interlockfehler, abgelehnter Moduswechsel und Verbindungsfehler beim Scharfstellen.

In [2]:
suite = unittest.defaultTestLoader.loadTestsFromModule(test_flare_nx)
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise AssertionError("Offline-Tests fehlgeschlagen.")

......................
----------------------------------------------------------------------
Ran 22 tests in 0.046s

OK


## Interlock und Ausgangszustand ablesen

Die Statuszelle einmal mit offenem Interlock ausführen und den Wert „Interlock offen“ mit dem Aufbau vergleichen. Dann den Interlock am Aufbau schließen und dieselbe Zelle nochmals ausführen. Bei „Interlock offen: False“ und „Interlockfehler: True“ kann der gespeicherte Fehler in der folgenden Reset-Zelle quittiert werden. Für den späteren Scharftest werden ein leeres Fehlerregister, Standby und ein inaktiver Triggereingang erwartet.

In [3]:
RUN_STATUS = True
if RUN_STATUS:
    with laser:
        print("Controller:", laser.query_parameter("*IDN?"))
        faults = laser.read_system_faults()
        status = laser.read_system_status()
        print(f"SYST:FAUL = 0x{faults:08X} · SYST:STAT = 0x{status:08X}")
        print("Interlock offen:", bool(faults & flare_nx.SystemFault.INTERLOCK_OPEN))
        print("Interlockfehler:", bool(faults & flare_nx.SystemFault.INTERLOCK_FAULT))
        print("Standby:", bool(status & flare_nx.SystemStatus.STANDBY))
        print("Triggereingang aktiv:", bool(status & flare_nx.SystemStatus.TRIGGER_INPUT))
        print("ENA:", laser.read_laser_enabled(), "· REM:", laser.read_remote_enabled())
        print("Pulsmodus:", laser.read_pulse_mode(), "· Pulszähler:", laser.read_pulse_count())
        print(f"Kopftemperatur: {laser.read_head_temperature_c():.3f} °C")

Controller: Coherent, Inc - FlareNX Controller - Serial Enter a Serial Number - V3.0.24 - 2021 Aug 30
SYST:FAUL = 0x00000000 · SYST:STAT = 0x00000304
Interlock offen: False
Interlockfehler: False
Standby: True
Triggereingang aktiv: False
ENA: False · REM: False
Pulsmodus: 3 · Pulszähler: 21322
Kopftemperatur: 26.294 °C


## Gespeicherten Interlockfehler quittieren

Nach dem Schließen des Interlocks diese Zelle einzeln aktivieren. Der Treiber prüft den aktuellen Fehlerstatus und verlangt ENA und REM auf OFF, bevor er die Quittierung nach OEM-Abschnitt 5.2.4 sendet. Die Anzeige danach enthält alle verbleibenden Controllerfehler. Anschließend die Statuszelle erneut ausführen, um auch Standby abzulesen.

In [5]:
RUN_INTERLOCK_RESET = True
if RUN_INTERLOCK_RESET and input("Interlock physisch geschlossen? Zum Quittieren RESET eingeben: ").strip() == "RESET":
    with laser:
        remaining = laser.reset_interlock_fault()
        print(f"Interlockfehler frei · aktuelles Fehlerregister: 0x{remaining:08X}")

Interlockfehler frei · aktuelles Fehlerregister: 0x00000000


## WaveGen elektrisch prüfen

Für diesen Test **Gen Out ausschließlich direkt mit CH1** am DSOX1204G verbinden, mit 1 MΩ Eingang und ohne zusätzlichen Abschluss. Die Triggerleitung zum Laser abziehen. Der Test erzeugt einen periodischen Pulszug mit 0–5 V und 50 µs Sollbreite. Er zeichnet eine frische Flanke auf CH1 auf und schaltet den Generator wieder aus.

Die Spannungen gelten für diese Last. Beim späteren Anschluss an den Laser die tatsächliche Last und den Ruhepegel am Triggereingang prüfen. Laut OEM-Handbuch, Abschnitt 6.3.2.3, ist vor der Moduswahl TTL-Low (typisch unter 0,4 V) erforderlich. Jeder externe +5-V-Puls muss mindestens 20 µs lang sein.

In [9]:
RUN_WAVEGEN_LOOPBACK = True
LEVEL_TOLERANCE_V = 0.25  # Acceptance band for this CH1 loopback measurement.
if RUN_WAVEGEN_LOOPBACK and input("Nur Gen Out → CH1 verbunden? Zur Bestätigung CH1 eingeben: ").strip() == "CH1":
    import numpy as np

    with devices["oscilloscope"] as scope:
        scope.configure_ttl(load="HIGHZ", low_v=0, high_v=5, frequency_hz=1, pulse_width_s=50e-6)
        scope.configure_channel(1, enabled=True, probe=1, coupling="DC", invert=False,
                                scale_v=1, offset_v=2.5)
        scope.configure_timebase(scale_s=20e-6, position_s=0)
        scope.configure_acquisition("NORMAL")
        scope.configure_edge_trigger(1, level_v=2.5)
        scope.arm_single()
        scope.wait_until_armed(timeout_s=5)
        scope.wavegen_output(True)
        scope.wait_for_single(timeout_s=5)
        scope.wavegen_output(False)
        wave = scope.read_waveforms((1,), points=10000)[1]

    if wave.missing.any() or wave.clipped_high.any() or wave.clipped_low.any():
        raise ValueError("CH1 enthält fehlende oder abgeschnittene Messwerte.")
    high = wave.voltage_v >= 2.5
    edges = np.flatnonzero(high[1:] != high[:-1]) + 1
    if len(edges) != 2 or high[0] or high[-1]:
        raise ValueError("Im Aufnahmefenster wurde kein einzelner vollständiger High-Puls erfasst.")
    width = wave.time_s[edges[1]] - wave.time_s[edges[0]]
    low_v = np.median(wave.voltage_v[~high])
    high_v = np.median(wave.voltage_v[high])
    print(f"CH1: Low {low_v:.2f} V · High {high_v:.2f} V · Breite bei 2,5 V: {width * 1e6:.1f} µs")
    if abs(low_v) > LEVEL_TOLERANCE_V or abs(high_v - 5) > LEVEL_TOLERANCE_V:
        raise ValueError(f"CH1-Pegel außerhalb der gewählten Messtoleranz ±{LEVEL_TOLERANCE_V} V.")
    if low_v >= 0.4 or width < 20e-6:
        raise ValueError("Ruhepegel oder Pulsbreite liegt außerhalb der Handbuchvorgabe.")
    print("Elektrische Prüfung an CH1 bestanden. WaveGen ist aus.")

CH1: Low -0.03 V · High 4.91 V · Breite bei 2,5 V: 50.0 µs
Elektrische Prüfung an CH1 bestanden. WaveGen ist aus.


## Modus 3 bei ausgeschaltetem Laser prüfen

Vor dem Verbinden den externen Triggereingang auf TTL-Low halten. Diese Zelle prüft die Konfiguration unabhängig vom Scharftest. Ist Modus 3 bereits aktiv, entfällt der Modusschreibbefehl.

In [4]:
RUN_MODE_TEST = True
if RUN_MODE_TEST:
    with laser:
        laser.configure_single_shot()
        mode, enabled = laser.read_pulse_mode(), laser.read_laser_enabled()
        print(f"Pulsmodus: {mode} · ENA: {enabled}")
        if mode != 3 or enabled:
            raise RuntimeError("Erwartet werden Modus 3 und ENA aus.")

Pulsmodus: 3 · ENA: False


## Einen extern ausgelösten Schuss prüfen

Nur am freigegebenen Laseraufbau ausführen. Vor der Bestätigung den Hardware-Interlock aktivieren, TTL-Low am angeschlossenen Triggereingang prüfen und die Quelle für **genau einen externen Puls** vorbereiten. Die Einzelausgabe muss separat am Aufbau nachgewiesen sein. Die WaveGen-Pulsfunktion im Scope-Treiber läuft periodisch. Die Taste *Single* am Oszilloskop begrenzt die Aufnahme. Kurzes Ein-/Ausschalten des Generators kann mehrere Flanken liefern. Solange nur der Pulszug verfügbar ist, die elektrische Prüfung oben verwenden.

Nach dem ENA-Befehl zeigt die Zelle ENA, Standby und die Controllerregister mit Zeitstempel an. Die blaue LED bezeichnet laut OEM-Tabelle 8-1 Standby. Die LED während des zehnsekündigen Triggerfensters beobachten. Den zuvor geprüften Einzelpuls in diesem Fenster extern auslösen. Bei Zeitablauf, Fehler, Zähleränderung oder Abbruch fordert der Treiber OFF an. Nach bestätigtem OFF ist eine blaue LED zu erwarten. Der Zählervergleich zeigt, wie viele Pulse der Controller registriert hat.

In [7]:
RUN_SINGLE_SHOT = True
if RUN_SINGLE_SHOT and input("Interlock freigegeben, TTL low und Einzelpulsquelle geprüft? SCHARF eingeben: ").strip() == "SCHARF":
    with laser:
        laser.configure_single_shot()
        before = laser.read_pulse_count()
        laser.arm_single_shot()
        start = time.monotonic()
        previous = None
        blocking = (flare_nx.SystemStatus.FAULT | flare_nx.SystemStatus.WARM_UP
                    | flare_nx.SystemStatus.TEMPERATURE_FAULT)
        while time.monotonic() - start < 10:
            enabled = laser.read_laser_enabled()
            status = laser.read_system_status()
            faults = laser.read_system_faults()
            standby = bool(status & flare_nx.SystemStatus.STANDBY)
            current = (enabled, status, faults)
            elapsed = time.monotonic() - start
            if current != previous:
                print(f"{elapsed:.1f} s · ENA={enabled} · Standby={standby} · "
                      f"SYST:STAT=0x{status:08X} · SYST:FAUL=0x{faults:08X}", flush=True)
            if not enabled or faults or status & blocking:
                raise RuntimeError("Controllerfreigabe entfallen oder Fehler/Warm-up gemeldet. Versuch abgebrochen.")
            if previous is None:
                print("Modus 3 · ENA=ON bestätigt. Triggerfenster für den externen Einzelpuls läuft.", flush=True)
            previous = current
            if laser.read_pulse_count() != before:
                break
            time.sleep(0.1)

    print("Laser OFF bestätigt.")
    with laser:
        after = laser.read_pulse_count()
        off_status = laser.read_system_status()
        print(f"Nach OFF: SYST:STAT=0x{off_status:08X} · Standby={bool(off_status & flare_nx.SystemStatus.STANDBY)}")
    delta = after - before
    print(f"Kopfzähler vorher {before} · nachher {after} · Differenz {delta}")
    if delta < 0:
        raise RuntimeError("Zählerstand rückläufig. Versuch nicht auswertbar.")
    if delta != 1:
        raise RuntimeError(f"Einzelpulstest fehlgeschlagen: {delta} Pulse gezählt.")
    print("Einzelpulstest bestanden: Der Kopfzähler hat genau einen Puls registriert.")

0.5 s · ENA=True · Standby=False · SYST:STAT=0x00000310 · SYST:FAUL=0x00000000
Modus 3 · ENA=ON bestätigt. Triggerfenster für den externen Einzelpuls läuft.
Laser OFF bestätigt.
Nach OFF: SYST:STAT=0x00000304 · Standby=True
Kopfzähler vorher 21325 · nachher 21326 · Differenz 1
Einzelpulstest bestanden: Der Kopfzähler hat genau einen Puls registriert.


**Herstellerunterlagen** · Coherent Flare NX OEM, 1306058 AE, Abschnitte 5.2.4 (Hardware-Interlock), 6.3.2.3 (Single Shot), 6.3.3 (Remote Enable), 6.3.7 (Pulszähler) und Tabellen 8-2/8-4. Die Befehle und Datenformate stehen in *User Command Set*, Blatt *User Commands*, Zeilen 4, 9, 15–18, 23, 27–29, 42 und 46.